# Neural Network Models — Iris Classification

This notebook demonstrates three neural network architectures on the **Iris dataset** (150 samples, 4 features, 3 classes).

Models covered:
1. **Perceptron** — single-layer, linear decision boundary
2. **Shallow MLP** — one hidden layer (32 units, ReLU)
3. **Deep MLP** — two hidden layers (64→32 units, Tanh)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

os.makedirs('figures', exist_ok=True)
RANDOM_STATE = 42
print('Libraries loaded.')

Libraries loaded.


## 1. Load & Preprocess Data

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = [f.replace(' (cm)', '') for f in iris.feature_names]
class_names   = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape},  Test: {X_test_sc.shape}')
print(f'Classes: {class_names}')

Train: (120, 4),  Test: (30, 4)
Classes: ['setosa' 'versicolor' 'virginica']


## 2. Data Overview

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
ax = axes[0]
vals, cnts = np.unique(y, return_counts=True)
bars = ax.bar(class_names, cnts, color=['#4C72B0','#DD8452','#55A868'],
              edgecolor='black', alpha=0.85)
for bar, c in zip(bars, cnts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            str(c), ha='center', va='bottom', fontweight='bold')
ax.set_title('Class Distribution', fontsize=13)
ax.set_ylabel('Count')

# Scaled feature boxplots
ax = axes[1]
pd.DataFrame(X_train_sc, columns=feature_names).boxplot(ax=ax, grid=False)
ax.set_title('Scaled Feature Distributions (Train)', fontsize=13)
ax.set_ylabel('Standardised Value')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('figures/data_overview.png', dpi=150)
plt.show()
print('Saved figures/data_overview.png')

Saved figures/data_overview.png


## 3. Model 1 — Perceptron

The **Perceptron** is the simplest neural network: a single layer with no hidden units and a hard threshold activation. It can only learn linearly separable patterns.

In [4]:
perceptron = Perceptron(max_iter=1000, random_state=RANDOM_STATE)
perceptron.fit(X_train_sc, y_train)
y_pred_p = perceptron.predict(X_test_sc)
acc_p = accuracy_score(y_test, y_pred_p)

print(f'Perceptron Test Accuracy: {acc_p:.4f}')
print(classification_report(y_test, y_pred_p, target_names=class_names))

Perceptron Test Accuracy: 0.8667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.88      0.70      0.78        10
   virginica       0.75      0.90      0.82        10

    accuracy                           0.87        30
   macro avg       0.88      0.87      0.87        30
weighted avg       0.88      0.87      0.87        30



## 4. Model 2 — Shallow MLP (1 Hidden Layer)

A **Multi-Layer Perceptron** adds one hidden layer with non-linear activations (ReLU), enabling it to learn non-linear decision boundaries.

In [5]:
mlp_shallow = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=RANDOM_STATE
)
mlp_shallow.fit(X_train_sc, y_train)
y_pred_s = mlp_shallow.predict(X_test_sc)
acc_s = accuracy_score(y_test, y_pred_s)

print(f'Shallow MLP Test Accuracy: {acc_s:.4f}')
print(f'Converged in {mlp_shallow.n_iter_} iterations')
print(classification_report(y_test, y_pred_s, target_names=class_names))

Shallow MLP Test Accuracy: 0.9333
Converged in 500 iterations
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



## 5. Model 3 — Deep MLP (2 Hidden Layers)

Adding a second hidden layer increases model capacity. We use **Tanh** activation for comparison.

In [6]:
mlp_deep = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='tanh',
    solver='adam',
    max_iter=500,
    random_state=RANDOM_STATE
)
mlp_deep.fit(X_train_sc, y_train)
y_pred_d = mlp_deep.predict(X_test_sc)
acc_d = accuracy_score(y_test, y_pred_d)

print(f'Deep MLP Test Accuracy: {acc_d:.4f}')
print(f'Converged in {mlp_deep.n_iter_} iterations')
print(classification_report(y_test, y_pred_d, target_names=class_names))

Deep MLP Test Accuracy: 0.9667
Converged in 312 iterations
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



## 6. Training Loss Curves

In [7]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(mlp_shallow.loss_curve_, label='Shallow MLP (ReLU, 32)',   color='#DD8452', linewidth=2)
ax.plot(mlp_deep.loss_curve_,    label='Deep MLP (Tanh, 64→32)',   color='#55A868', linewidth=2)

ax.set_title('Training Loss Curve', fontsize=13)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/loss_curves.png', dpi=150)
plt.show()
print('Saved figures/loss_curves.png')

Saved figures/loss_curves.png


## 7. Confusion Matrices

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y_pred) in zip(axes, [
    ('Perceptron',  y_pred_p),
    ('Shallow MLP', y_pred_s),
    ('Deep MLP',    y_pred_d),
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='gray')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}  (Acc={acc:.2f})', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('figures/confusion_matrices.png', dpi=150)
plt.show()
print('Saved figures/confusion_matrices.png')

Saved figures/confusion_matrices.png


## 8. Model Comparison

In [9]:
# Summary table
results = pd.DataFrame({
    'Model':        ['Perceptron', 'Shallow MLP', 'Deep MLP'],
    'Architecture': ['4 → 3', '4 → 32 → 3', '4 → 64 → 32 → 3'],
    'Activation':   ['Step', 'ReLU', 'Tanh'],
    'Accuracy':     [f'{acc_p:.4f}', f'{acc_s:.4f}', f'{acc_d:.4f}']
})
print(results.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
accs   = [acc_p, acc_s, acc_d]
labels = ['Perceptron', 'Shallow MLP\n(ReLU, 32)', 'Deep MLP\n(Tanh, 64-32)']
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(labels, accs, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Test Accuracy')
ax.set_title('Model Accuracy Comparison', fontsize=13)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, label='Perfect')
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/model_comparison.png', dpi=150)
plt.show()
print('Saved figures/model_comparison.png')

      Model    Architecture Activation Accuracy
 Perceptron           4 → 3       Step   0.8667
Shallow MLP      4 → 32 → 3       ReLU   0.9333
   Deep MLP 4 → 64 → 32 → 3       Tanh   0.9667


Saved figures/model_comparison.png


## 9. Decision Boundaries (PCA 2D Projection)

In [10]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

models_2d = [
    ('Perceptron',          Perceptron(max_iter=1000, random_state=RANDOM_STATE)),
    ('Shallow MLP (ReLU)',  MLPClassifier(hidden_layer_sizes=(32,), activation='relu',
                                          max_iter=500, random_state=RANDOM_STATE)),
    ('Deep MLP (Tanh)',     MLPClassifier(hidden_layer_sizes=(64,32), activation='tanh',
                                          max_iter=500, random_state=RANDOM_STATE)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cmap_pts = {0: '#4C72B0', 1: '#DD8452', 2: '#55A868'}

for ax, (name, model) in zip(axes, models_2d):
    model.fit(X_train_2d, y_train)
    x0min, x0max = X_train_2d[:,0].min()-0.5, X_train_2d[:,0].max()+0.5
    x1min, x1max = X_train_2d[:,1].min()-0.5, X_train_2d[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x0min, x0max, 250),
                         np.linspace(x1min, x1max, 250))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.20, cmap='tab10')
    for cls in range(3):
        mask = y_train == cls
        ax.scatter(X_train_2d[mask,0], X_train_2d[mask,1],
                   color=cmap_pts[cls], s=30, edgecolor='k',
                   linewidth=0.4, alpha=0.85, label=class_names[cls])
    test_acc = accuracy_score(y_test, model.predict(X_test_2d))
    ax.set_title(f'{name}\n(2D Acc={test_acc:.2f})', fontsize=11)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8)

plt.suptitle('Decision Boundaries (PCA 2D Projection)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figures/decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figures/decision_boundaries.png')

Saved figures/decision_boundaries.png


## Summary

| Model | Architecture | Activation | Notes |
|-------|-------------|------------|-------|
| Perceptron | 4 → 3 | Step | Linear boundaries only |
| Shallow MLP | 4 → 32 → 3 | ReLU | Captures non-linear patterns |
| Deep MLP | 4 → 64 → 32 → 3 | Tanh | More expressive, may overfit on small data |

**Key takeaways:**
- The Perceptron is limited to linear decision boundaries.
- Adding hidden layers with non-linear activations lets MLPs model complex boundaries.
- On the small Iris dataset, a single hidden layer is typically sufficient.